In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

# --- 1. 加载模型和分词器 ---
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./qwen_lora_adapter"   # LoRA适配器保存路径

print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔑 QLoRA的关键: 使用4-bit量化加载基础模型，大幅降低显存占用
# 如果显存充足（例如8GB以上），可以将 load_in_4bit 设为 False，切换为标准 LoRA
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    load_in_4bit=True,           # 👈 启用4-bit量化，这是QLoRA的核心
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# --- 2. 配置LoRA ---
# target_modules 在不同模型架构中名称不同
# 对于Qwen模型，常用目标模块包括 "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
lora_config = LoraConfig(
    r=8,                        # 低秩矩阵的秩，常用值4-16[reference:5][reference:6]
    lora_alpha=16,              # 缩放系数
    target_modules=["q_proj", "v_proj"],  # 作用于注意力层的投影矩阵[reference:7]
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 查看可训练参数量

# --- 3. 准备数据 ---
# 假设数据集格式为JSON, 每行包含 "instruction" 和 "output" 字段
dataset = load_dataset('json', data_files='train_data.json', split='train')

def preprocess_function(examples):
    texts = [f"Instruction: {inst}\nResponse: {out}" 
             for inst, out in zip(examples["instruction"], examples["output"])]
    tokenized = tokenizer(texts, truncation=True, padding="max_length", max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=["instruction", "output"])

# --- 4. 配置训练参数 ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    fp16=True,                  # 启用混合精度训练
    logging_steps=10,
    save_steps=50,
    max_steps=200,              # 小数据集示例，可调大
    report_to="none",
    remove_unused_columns=True,
    label_names=["labels"],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

# --- 5. 开始微调和保存 ---
print("Starting training...")
trainer.train()

print("Saving LoRA adapter...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# --- 6. 可选: 将LoRA权重合并回基础模型（便于推理部署）---
# 这一步会生成一个完整的模型文件，直接用标准的transformers就能加载，不再依赖peft
print("Merging and saving full model...")
merged_model = model.merge_and_unload()  # 👈 合并LoRA权重
merged_model.save_pretrained(OUTPUT_DIR + "_merged")
tokenizer.save_pretrained(OUTPUT_DIR + "_merged")
print(f"Done! Merged model saved to {OUTPUT_DIR}_merged")